In [1]:
import sys
sys.path.insert(1, '/home/santolam/MyPythonLibrary/MAR-RECAP/libmar/')
import climbasis as climb
from climbasis import *
import domain as dom
import myplot
import numpy.ma as ma
from myplot import *
import obsinfo as obs
from obsinfo import *



In [2]:
plotsDir='/home/santolam/MyPythonLibrary/MAR-RECAP/scripts_paper_on_evaluation_MAR/figures_paper_on_evaluation/'

In [3]:
####
##Donnes meteo
sourceData='/bettik/PROJECTS/pr-regional-climate/santolam/glacioclim/donnees_precip/'
#
#Precipitation_Khare_corrected_Nov2016_Nov2023_final.xlsx
#Precipitation_Pyramid_Pheriche_corrected_Apr2016_Nov2023_final.xlsx
#Precipitation_Pyramid_Pheriche_corrected_Dec2012_Apr2016_final.xlsx

In [4]:
#https://github.com/Andeol-ramb/MAR_HMA_GLACIOCLIM_Comparisons/blob/main/Sensitivity_Test_MAR/Sensitivity_Test_Pyramide_Precip.ipynb
####
## Opening Pheriche+Khare
####

sourceDataSta=sourceData#'/Users/santolam/Data/glacioclim/donnees_precip/'
path1 =sourceDataSta+"/Precipitation_Pyramid_Pheriche_corrected_Dec2012_Apr2016_final.xlsx"
xl_data1=pd.ExcelFile(path1)

dfs1= {sheet_name: xl_data1.parse(sheet_name) 
       for sheet_name in xl_data1.sheet_names}

DataPyramid_Pheriche12_16=dfs1['Precip_data_Dec12_Apr16']

NameVarsPP12_16=['TimeY','RRYhourly','RRYcumul','TimeH','RRHhourly','RRHcumul']
UnitVarsPP16_23=['data/hour','mm','mm','data/hour','mm','mm',]

dfile1P12_16= pd.DataFrame(data=DataPyramid_Pheriche12_16.loc[10:])
df1P12_16=pd.DataFrame(dfile1P12_16.values, columns=NameVarsPP12_16)

df1P12_16['TimeY'] = df1P12_16['TimeY'].dt.round('h')

path =sourceDataSta+"/Precipitation_Pyramid_Pheriche_corrected_Apr2016_Nov2023_final.xlsx"
xl_data=pd.ExcelFile(path)

dfs = {sheet_name: xl_data.parse(sheet_name) 
       for sheet_name in xl_data.sheet_names}
DataPyramid_Pheriche16_23=dfs['Precip_data_Apr16_Nov23']

NameVarsPP16_23=['TimeY','TairY','uY','RHY','WMOphaseY','RRYhourly','RRYcumul','TimeH','TairH','uH','RHh','WMOphaseH','RRHhourly','RRHcumul']
UnitVarsPP16_23=['data/hour','(°C)','(ms-1)','(%)','mm','mm','data/hour','(°C)','(ms-1)','(%)','mm','mm',]

dfile1P16_23= pd.DataFrame(data=DataPyramid_Pheriche16_23.loc[10:57630]) #3399 marche, Toujours la même taille jusque 3399 puis toujours lenght var=t-1 par la suite 
#CHANGEMENT DANS TIME:3389 	2016-09-15 17:59:59.!999! / 2016-09-15 18:59:59.!998! 
#donc erreur à chaque changement de décimale, corriger cette data avant tout autre chose


dfile2P16_23=pd.DataFrame(data=DataPyramid_Pheriche16_23.loc[57632:]) 
df1P16_23=pd.DataFrame(dfile1P16_23.values, columns=NameVarsPP16_23)
df2P16_23=pd.DataFrame(dfile2P16_23.values, columns=NameVarsPP16_23)
df1P16_23['TimeY'] = df1P16_23['TimeY'].dt.round('h')
df2P16_23['TimeY'] = df2P16_23['TimeY'].dt.round('30min')

#Definition d'une fréquence permettant par la suite le traitement avec des Xarray
def NewTime(df, NameVar, values, freq):
    time = pd.to_datetime(df['TimeY'].values)
    ds = xr.Dataset({NameVar: ("time", np.asarray(values))}, coords={"time": time})
    return ds
# Time step: 1h

RR1_PY=[float(thing) for thing in df1P12_16['RRYhourly'].values]
RRY1_PY = [float(thing) for thing in df1P16_23['RRYhourly'].values]
dsY1_PY = NewTime(df1P16_23, 'RRYhourly', RRY1_PY, '1h')
dsRR1_PY= NewTime(df1P12_16, 'RRYhourly', RR1_PY, '1h')

# Time step : 30 min 
RRY2_PY = [float(thing) for thing in df2P16_23['RRYhourly'].values]
dsY2_PY = NewTime(df2P16_23, 'RRYhourly', RRY2_PY, '30min')
RR_PY = xr.concat([dsRR1_PY,dsY1_PY, dsY2_PY], dim='time')


RR1_PH=[float(thing) for thing in df1P12_16['RRHhourly'].values]
RRY1_PH = [float(thing) for thing in df1P16_23['RRHhourly'].values]
dsY1_PH = NewTime(df1P16_23, 'RRHhourly', RRY1_PH, '1h')
dsRR1_PH= NewTime(df1P12_16, 'RRHhourly', RR1_PH, '1h')

# Time step : 30 min 
RRY2_PH = [float(thing) for thing in df2P16_23['RRHhourly'].values]
dsY2_PH = NewTime(df2P16_23, 'RRHhourly', RRY2_PH, '30min')
RR_PH = xr.concat([dsRR1_PH,dsY1_PH, dsY2_PH], dim='time')

#####
#
#Khare
#
#####
fileName_KH='Precipitation_Khare_corrected_Nov2016_Nov2023_final.xlsx'
xl_file_KH = pd.ExcelFile(sourceDataSta+fileName_KH)
print(xl_file_KH)
#
dfs_KH = {sheet_name: xl_file_KH.parse(sheet_name) 
          for sheet_name in xl_file_KH.sheet_names}

NameVars_KH=['Time','Tair' ,'u' ,'RH', 'Phase_WMO','RR_WMO_Half-hourly_values','RR_WMO_Khare_cumulated']
UnitVars_KH=['time','(°C)','(ms-1)','(%)','phase','mm w.e.','mm w.e.']

##Define new DataFrame and changing name of columns
dfile_KH = pd.DataFrame(data=dfs_KH["Precip_data_Nov16_Nov23"].loc[8:])
df_KH=pd.DataFrame(dfile_KH.values, columns=NameVars_KH)

path =sourceDataSta+fileName_KH
xl_data=pd.ExcelFile(path)
dfs = {sheet_name: xl_data.parse(sheet_name) 
       for sheet_name in xl_data.sheet_names}
DataKhare16_23=dfs['Precip_data_Nov16_Nov23']
NameVarsPP16_23=['Time','Tair','u','RH','WMOphase','RRhourly','RRcumul']
UnitVarsPP16_23=['data/hour','(°C)','(ms-1)','(%)','mm','mm']
dfileP16_23= pd.DataFrame(data=DataKhare16_23.loc[10:])
dfP16_23=pd.DataFrame(dfileP16_23.values, columns=NameVarsPP16_23)

RR_KH=[float(i) for i in dfP16_23.RRhourly.values] 
def NewTime(df, NameVar, values, freq):
    time = pd.to_datetime(df['Time'].values)
    ds = xr.Dataset({NameVar: ("time", np.asarray(values))}, coords={"time": time})
    return ds
RR_KH=NewTime(dfP16_23,'RRhourly',RR_KH,'30min')
#RR=RR.sel(time=slice('2016'))


rr_daily_PY=RR_PY.resample(time='D').sum()
rr_daily_PY=rr_daily_PY.RRYhourly

rr_daily_PH=RR_PH.resample(time='D').sum()
rr_daily_PH=rr_daily_PH.RRHhourly

rr_daily_KH=RR_KH.resample(time='D').sum()
rr_daily_KH=rr_daily_KH.RRhourly

idatePY = str(rr_daily_PY.time[0].values)[:10]
fdatePY = str(rr_daily_PY.time[-1].values)[:10]
idatePH = str(rr_daily_PH.time[0].values)[:10]
fdatePH = str(rr_daily_PH.time[-1].values)[:10]
idateKH = str(rr_daily_KH.time[0].values)[:10]
fdateKH = str(rr_daily_KH.time[-1].values)[:10]


In [5]:
###
## PYRAMID Salerno
### 

In [6]:
sourceData='/bettik/PROJECTS/pr-regional-climate/santolam/PYRAMID_NETWORK_DATA_v3/'

file_AWS0='AWS0_Z5035_Lvl1.csv'
file_AWS1='AWS1_Z5035_Lvl1.csv'
file_AWS2='AWS2_Z4260_Lvl1.csv'
file_AWS3='AWS3_Z2660_Lvl1.csv'
file_AWS4='AWS4_Z5600_Lvl1.csv'
file_AWS5='AWS5_Z3570_Lvl1.csv'
file_AWSSC='AWSSC_Z7986_Lvl1.csv'
file_CNG='CNG_SNP_Z5700_Lvl1.csv'
file_mon_Pyr='Pyramid_Z5035_lvl2.csv'

# Store filenames in a dictionary for easy looping
files = {
    'AWS0': file_AWS0,
    'AWS1': file_AWS1,
    'AWS2': file_AWS2,
    'AWS3': file_AWS3,
    'AWS4': file_AWS4,
    'AWS5': file_AWS5,
    'AWSSC': file_AWSSC,
    'CNG': file_CNG,
    #'Pyramid': file_mon_Pyr
}
dfs = {}
for name, fname in files.items():
    path = os.path.join(sourceData, fname)
    
    # Read semicolon-delimited CSV
    df = pd.read_csv(path, sep=';')
    
    # Combine into datetime
    df['TIMESTAMP'] = pd.to_datetime(dict(
        year=df['year'],
        month=df['month'],
        day=df['day'],
        hour=df['hour']
    ))
    
    # Set datetime index and drop date columns
    df.set_index('TIMESTAMP', inplace=True)
    df.drop(columns=['year', 'month', 'day', 'hour'], inplace=True)
    
    # Replace physically impossible values (< -100) with NaN
    df[df < -100] = np.nan
    
    # Store
    dfs[name] = df
    
# Optional: unpack for direct access
df_AWS0  = dfs['AWS0']
df_AWS1  = dfs['AWS1']
df_AWS2  = dfs['AWS2']
df_AWS3  = dfs['AWS3']
df_AWS4  = dfs['AWS4']
df_AWS5  = dfs['AWS5']
df_AWSSC = dfs['AWSSC']
df_CNG   = dfs['CNG']
#df_Pyr   = dfs['Pyramid']

# Quick check
print(df_AWS0.head())

# Daily total precipitation (sum of hourly rates)
rr_AWS0_daily = df_AWS0['RR'].resample('D').sum()
rr_AWS1_daily = df_AWS1['RR'].resample('D').sum()
rr_AWS2_daily = df_AWS2['RR'].resample('D').sum()
rr_AWS3_daily = df_AWS3['RR'].resample('D').sum()
rr_AWS4_daily = df_AWS4['RR'].resample('D').sum()
rr_AWS5_daily = df_AWS5['RR'].resample('D').sum()


                       AT   RR     AP    RH    WS  WD
TIMESTAMP                                            
1994-01-01 02:00:00 -11.9  0.0  551.4  25.0  0.00 NaN
1994-01-01 04:00:00 -13.2  0.0  551.0  23.0  0.00 NaN
1994-01-01 06:00:00 -10.5  0.0  551.1  10.0  0.26 NaN
1994-01-01 08:00:00 -10.9  0.0  551.9  17.0  0.00 NaN
1994-01-01 10:00:00  -7.1  0.0  553.1  17.0  5.58 NaN


In [7]:
sourceData='/bettik/PROJECTS/pr-regional-climate/santolam/nat_geo_stations_everest/'
#sourceData='/Users/santolam/Desktop/nat_geo_stations_everest/'
##
file_bc='Base_Camp_20230630.csv'
file_c2='Camp2_20230630.csv'
file_phort='Phortse_20230630.csv'
file_bish='Bishop_Rock_20230630.csv'
file_sc='South_Col_20230630.csv'

df_bc = pd.read_csv(os.path.join(sourceData, file_bc), parse_dates=['TIMESTAMP'])
df_c2 = pd.read_csv(os.path.join(sourceData, file_c2), parse_dates=['TIMESTAMP'])
df_phort = pd.read_csv(os.path.join(sourceData, file_phort), parse_dates=['TIMESTAMP'])
df_bish = pd.read_csv(os.path.join(sourceData, file_bish), parse_dates=['TIMESTAMP'])
df_sc = pd.read_csv(os.path.join(sourceData, file_sc), parse_dates=['TIMESTAMP'])

# Optional: set timestamp as index
for df in [df_bc, df_c2, df_phort, df_bish, df_sc]:
    df.set_index('TIMESTAMP', inplace=True)

# Quick check
print(df_bc.head())

rr_bc_daily = df_bc['PRECIP'].resample('D').sum()
rr_phort_daily = df_phort['PRECIP'].resample('D').sum()


                     T_HMP      RH   PRESS  PRECIP
TIMESTAMP                                         
2019-10-17 13:00:00 -4.078  99.578  536.71     0.0
2019-10-17 14:00:00 -4.282  92.273  537.15     0.0
2019-10-17 15:00:00 -3.584  86.358  537.28     0.0
2019-10-17 16:00:00 -3.458  88.629  537.25     0.0
2019-10-17 17:00:00 -3.966  74.045  537.22     0.0


In [8]:
###Coordinates

In [9]:
import pandas as pd
from pyproj import Transformer

# Define the transformer (UTM Zone 45N → WGS84)
transformer = Transformer.from_crs("epsg:32645", "epsg:4326", always_xy=True)

# --- Coordinates conversions (you already have some precomputed) ---
# MN (Mera North)
eastingMN, northingMN = 489885, 3065938
lon_MN, lat_MN = transformer.transform(eastingMN, northingMN)
elev_MN = 5360

# CH (Changri)
lon_CH, lat_CH, elev_CH = 86.77948450, 27.98336311, 5360

# ML (Merala)
lon_ML, lat_ML, elev_ML = 86.900, 27.735, 5352.008

# KH (Khare)
lon_KH, lat_KH, elev_KH = 86.8776142768303, 27.74424682282522, 4888

# PH (Pheriche)
lon_PH, lat_PH, elev_PH = 86.81972903284355, 27.895048949594884, 4260

# PY (Pyramid)
lon_PY, lat_PY, elev_PY = 86.81296905961787, 27.957821815535848, 5035

# MS (Mera South)
eastingMS, northingMS = 487553.653, 3064704.231
lon_MS, lat_MS = transformer.transform(eastingMS, northingMS)
elev_MS = 5360  # estimated same as MN, if not otherwise specified

# MC (Mera Central)
eastingMC, northingMC = 488409.424, 3066148.049
lon_MC, lat_MC = transformer.transform(eastingMC, northingMC)
elev_MC = 5770

# --- Define metadata list ---
stations_info_glacioclim = [
    {"ID": "MN", "Name": "Mera Naulek", "Lat": lat_MN, "Lon": lon_MN, "Elev_m": elev_MN, "Timestep": "1h"},
    {"ID": "CH", "Name": "Changri", "Lat": lat_CH, "Lon": lon_CH, "Elev_m": elev_CH, "Timestep": "1h"},
    {"ID": "ML", "Name": "Mera La", "Lat": lat_ML, "Lon": lon_ML, "Elev_m": elev_ML, "Timestep": "1h"},
    {"ID": "KH", "Name": "Khare", "Lat": lat_KH, "Lon": lon_KH, "Elev_m": elev_KH, "Timestep": "1h"},
    {"ID": "PH", "Name": "Pheriche", "Lat": lat_PH, "Lon": lon_PH, "Elev_m": elev_PH, "Timestep": "1h"},
    {"ID": "PY", "Name": "Pyramid", "Lat": lat_PY, "Lon": lon_PY, "Elev_m": elev_PY, "Timestep": "1h"},
    {"ID": "MS", "Name": "Mera Summir", "Lat": lat_MS, "Lon": lon_MS, "Elev_m": elev_MS, "Timestep": "1h"},
    {"ID": "MC", "Name": "Mera Camp 5770", "Lat": lat_MC, "Lon": lon_MC, "Elev_m": elev_MC, "Timestep": "1h"},
]

# --- Create DataFrame ---
df_stations_glacioclim = pd.DataFrame(stations_info_glacioclim)

# Display neatly
print(df_stations_glacioclim.to_string(index=False, justify='center', col_space=12))


     ID           Name           Lat          Lon        Elev_m      Timestep  
     MN         Mera Naulek  27.717723    86.897390     5360.000        1h     
     CH             Changri  27.983363    86.779484     5360.000        1h     
     ML             Mera La  27.735000    86.900000     5352.008        1h     
     KH               Khare  27.744247    86.877614     4888.000        1h     
     PH            Pheriche  27.895049    86.819729     4260.000        1h     
     PY             Pyramid  27.957822    86.812969     5035.000        1h     
     MS         Mera Summir  27.706566    86.873753     5360.000        1h     
     MC      Mera Camp 5770  27.719608    86.882420     5770.000        1h     


In [10]:
##
# --- AWS and related station metadata ---

# AWSSC (South Col)
lat_AWSSC, lon_AWSSC, elev_AWSSC = 27.98, 86.76, 7986

# CNG_SNP (Changri Nup)
lat_CNG, lon_CNG, elev_CNG = 27.96, 86.93, 5700

# AWS0 (Pyramid)
lat_AWS0, lon_AWS0, elev_AWS0 = 27.96, 86.81, 5035

# AWS1 (Pyramid)
lat_AWS1, lon_AWS1, elev_AWS1 = 27.96, 86.81, 5035

# AWS2 (Pheriche)
lat_AWS2, lon_AWS2, elev_AWS2 = 27.90, 86.82, 4260

# AWS3 (Lukla)
lat_AWS3, lon_AWS3, elev_AWS3 = 27.70, 86.72, 2660

# AWS4 (Kala Patthar)
lat_AWS4, lon_AWS4, elev_AWS4 = 27.99, 86.83, 5600

# AWS5 (Namche)
lat_AWS5, lon_AWS5, elev_AWS5 = 27.80, 86.71, 3570

# Pyramid (monthly level 2)
lat_Pyr, lon_Pyr, elev_Pyr = 27.96, 86.81, 5035

# Define metadata as a list of dicts
stations_info_salerno = [
    {"ID": "AWSSC", "Name": "South Col", "Lat": 27.98, "Lon": 86.76, "Elev_m": 7986, "Timestep": "1h"},
    {"ID": "CNG_SNP", "Name": "Changri Nup", "Lat": 27.96, "Lon": 86.93, "Elev_m": 5700, "Timestep": "1h"},
    {"ID": "AWS0", "Name": "Pyramid", "Lat": 27.96, "Lon": 86.81, "Elev_m": 5035, "Timestep": "1h"},
    {"ID": "AWS1", "Name": "Pyramid", "Lat": 27.96, "Lon": 86.81, "Elev_m": 5035, "Timestep": "1h"},
    {"ID": "AWS2", "Name": "Pheriche", "Lat": 27.90, "Lon": 86.82, "Elev_m": 4260, "Timestep": "1h"},
    {"ID": "AWS3", "Name": "Lukla", "Lat": 27.70, "Lon": 86.72, "Elev_m": 2660, "Timestep": "1h"},
    {"ID": "AWS4", "Name": "Kala Patthar", "Lat": 27.99, "Lon": 86.83, "Elev_m": 5600, "Timestep": "1h"},
    {"ID": "AWS5", "Name": "Namche", "Lat": 27.80, "Lon": 86.71, "Elev_m": 3570, "Timestep": "1h"},
    {"ID": "PYmon", "Name": "Pyramid (monthly)", "Lat": 27.96, "Lon": 86.81, "Elev_m": 5035, "Timestep": "1month"},
]

# Create DataFrame
df_stations_salerno = pd.DataFrame(stations_info_salerno)

# Display
print(df_stations_salerno)

        ID               Name    Lat    Lon  Elev_m Timestep
0    AWSSC          South Col  27.98  86.76    7986       1h
1  CNG_SNP        Changri Nup  27.96  86.93    5700       1h
2     AWS0            Pyramid  27.96  86.81    5035       1h
3     AWS1            Pyramid  27.96  86.81    5035       1h
4     AWS2           Pheriche  27.90  86.82    4260       1h
5     AWS3              Lukla  27.70  86.72    2660       1h
6     AWS4       Kala Patthar  27.99  86.83    5600       1h
7     AWS5             Namche  27.80  86.71    3570       1h
8    PYmon  Pyramid (monthly)  27.96  86.81    5035   1month


In [11]:
#Nat geo
#BASE CAMP AWS (27.9952 N, 86.8406 E, 5315 M ASL)
lat_bc,lon_bs,elev_bc=27.9952, 86.8406, 5315
#CAMP II AWS (27.9810 N, 86.9023 E, 6464 M ASL)
lat_c2,lon_c2,elev_c2=27.9810, 86.9023, 6464 
#PHORTSE AWS (27.8456 N, 86.7472 E, 3810 M ASL)
lat_phort,lon_phort,elev_phort=27.8456, 86.7472, 3810 
#BISHOP ROCK AWS (27.9735 N, 86.9308 E, 8810 M ASL)
lat_bish,lon_bish,elev_bish=27.9735, 86.9308, 8810 
#MT. EVEREST SOUTH COL AWS (27.9719 N, 86.9295 E, 7945 M ASL)
lat_sc,lon_sc,elev_sc=27.9719, 86.9295, 7945


# Define metadata for Everest AWS network
stations_info_everest = [
    {"ID": "bc", "Name": "BaseCampAWS", "Lat": 27.9952, "Lon": 86.8406, "Elev_m": 5315},
    {"ID": "c2", "Name": "Camp II AWS", "Lat": 27.9810, "Lon": 86.9023, "Elev_m": 6464},
    {"ID": "phort", "Name": "Phortse AWS", "Lat": 27.8456, "Lon": 86.7472, "Elev_m": 3810},
    {"ID": "bc", "Name": "Bishop Rock AWS", "Lat": 27.9735, "Lon": 86.9308, "Elev_m": 8810},
    {"ID": "sc", "Name": "Mt. Everest South Col AWS", "Lat": 27.9719, "Lon": 86.9295, "Elev_m": 7945},
]

# Create DataFrame
df_everest_stations = pd.DataFrame(stations_info_everest)

# Display
print(df_everest_stations)

      ID                       Name      Lat      Lon  Elev_m
0     bc                BaseCampAWS  27.9952  86.8406    5315
1     c2                Camp II AWS  27.9810  86.9023    6464
2  phort                Phortse AWS  27.8456  86.7472    3810
3     bc            Bishop Rock AWS  27.9735  86.9308    8810
4     sc  Mt. Everest South Col AWS  27.9719  86.9295    7945


In [12]:
# --- Merge all station metadata into one unified dictionary ---
stations_all = {}

# Add Salerno AWS
for sta in stations_info_salerno:
    stations_all[sta["ID"]] = (sta["Lat"], sta["Lon"], "SalernoAWS")

# Add Glacioclim AWS
for sta in stations_info_glacioclim:
    stations_all[sta["ID"]] = (sta["Lat"], sta["Lon"], "Glacioclim")

# Add NatGeo / Everest AWS
for sta in stations_info_everest:
    stations_all[sta["ID"]] = (sta["Lat"], sta["Lon"], "NatGeo")

# Check summary
print(f"Total stations merged: {len(stations_all)}")
print(stations_all.keys())

Total stations merged: 21
dict_keys(['AWSSC', 'CNG_SNP', 'AWS0', 'AWS1', 'AWS2', 'AWS3', 'AWS4', 'AWS5', 'PYmon', 'MN', 'CH', 'ML', 'KH', 'PH', 'PY', 'MS', 'MC', 'bc', 'c2', 'phort', 'sc'])


In [13]:
## Salerno
# Daily total precipitation (sum of hourly rates)
#rr_AWS0_daily = df_AWS0['RR'].resample('D').sum()
#rr_AWS1_daily = df_AWS1['RR'].resample('D').sum()
#rr_AWS2_daily = df_AWS2['RR'].resample('D').sum()
#rr_AWS3_daily = df_AWS3['RR'].resample('D').sum()
#rr_AWS4_daily = df_AWS4['RR'].resample('D').sum()
#rr_AWS5_daily = df_AWS5['RR'].resample('D').sum()
#jAWS0, iAWS0 
# --- Define date ranges for each AWS based on observations ---
idate_AWS0 = str(rr_AWS0_daily.index[0].date())
fdate_AWS0 = str(rr_AWS0_daily.index[-1].date())
idate_AWS1 = str(rr_AWS1_daily.index[0].date())
fdate_AWS1 = str(rr_AWS1_daily.index[-1].date())
idate_AWS2 = str(rr_AWS2_daily.index[0].date())
fdate_AWS2 = str(rr_AWS2_daily.index[-1].date())
idate_AWS3 = str(rr_AWS3_daily.index[0].date())
fdate_AWS3 = str(rr_AWS3_daily.index[-1].date())
idate_AWS4 = str(rr_AWS4_daily.index[0].date())
fdate_AWS4 = str(rr_AWS4_daily.index[-1].date())
idate_AWS5 = str(rr_AWS5_daily.index[0].date())
fdate_AWS5 = str(rr_AWS5_daily.index[-1].date())



In [14]:
##Nat geo
##
#rr_bc_daily = df_bc['PRECIP'].resample('D').sum()
#rr_phort_daily = df_phort['PRECIP'].resample('D').sum()
idate_bc     = str(rr_bc_daily.index[0].date())
fdate_bc     = str(rr_bc_daily.index[-1].date())

idate_phort  = str(rr_phort_daily.index[0].date())
fdate_phort  = str(rr_phort_daily.index[-1].date())


In [15]:

# --- Observed daily precipitation series ---
stations_rr_daily_glacioclim = {
    "Pheriche": rr_daily_PH,
    "Khare": rr_daily_KH,
    "Pyramid": rr_daily_PY
}



In [ ]:
date_ranges_glacioclim = {
    "MeraLa": f"{idatePH} to {fdatePH}",
    "MeraSummit": f"{idatePY} to {fdateMS}",
    "MeraNaulek": f"{idateMN} to {fdateMN}",
    "Pyramid": f"{idatePYG} to {fdatePYG}",
    "ChangriA3": f"{idateCHA3} to {fdateCHA3}"
}

In [16]:
# --- Observed daily total precipitation ---
stations_rr_daily_salerno = {
    "AWS0": rr_AWS0_daily,
    "AWS1": rr_AWS1_daily,
    "AWS2": rr_AWS2_daily,
    "AWS3": rr_AWS3_daily,
    "AWS4": rr_AWS4_daily,
    "AWS5": rr_AWS5_daily,
}



In [17]:

# --- Observed daily total precipitation ---
stations_rr_daily_natgeo = {
    "BaseCamp": rr_bc_daily,
    "Phortse": rr_phort_daily,
}



In [ ]:
# --- Create subplots (one per station) ---
fig, axes = plt.subplots(len(stations_rr_daily_glacioclim), 1, figsize=(18, 16), sharex=True)

# --- Plot each station ---
for ax, (name, obs_series) in zip(axes, stations_rr_daily_glacioclim.items()):
    # Observed daily temperature (line)
    ax.plot(obs_series.time.values, obs_series.values, color='red', lw=1.4, alpha=0.7, label="Observed")

      # Style + Title
    ax.set_ylabel("mm/day", rotation=0, labelpad=20)
    ax.set_title(f"{name} – Station",
                 loc='left', fontsize=11, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(frameon=False, loc='upper right', fontsize=9)

# Shared labels and figure title
axes[-1].set_xlabel("Date")
fig.suptitle("Daily Precipitation – Glacioclim AWS vs Model",
             fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.97])

plotname = f"support_FIGSX_timeseries_daily_pre.glacioclim"
plt.savefig(f"{plotsDir}{plotname}.png", bbox_inches="tight", dpi=300)
print(f" Saved: {plotsDir}{plotname}.png")

plt.show()


In [ ]:
date_ranges = {
    "MeraLa": f"{idateML} to {fdateML}",
    "MeraSummit": f"{idateMS} to {fdateMS}",
    "MeraNaulek": f"{idateMN} to {fdateMN}",
    "Pyramid": f"{idatePYG} to {fdatePYG}",
    "ChangriA3": f"{idateCHA3} to {fdateCHA3}"
}

In [ ]:
# --- Create subplots (one per station) ---
fig, axes = plt.subplots(len(stations_rr_daily_glacioclim), 1, figsize=(18, 16), sharex=True)

# --- Plot each station ---
for ax, (name, obs_series) in zip(axes, stations_rr_daily_salerno.items()):
    # Observed daily temperature (line)
    ax.plot(obs_series.time.values, obs_series.values, color='red', lw=1.4, alpha=0.7, label="Observed")

      # Style + Title
    ax.set_ylabel("mm/day", rotation=0, labelpad=20)
    ax.set_title(f"{name} – Station",
                 loc='left', fontsize=11, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(frameon=False, loc='upper right', fontsize=9)

# Shared labels and figure title
axes[-1].set_xlabel("Date")
fig.suptitle("Daily Precipitation – Ev-K2-CNR AWS vs Model",
             fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.97])

plotname = f"support_FIGSX_timeseries_daily_pre.salerno"
plt.savefig(f"{plotsDir}{plotname}.png", bbox_inches="tight", dpi=300)
print(f" Saved: {plotsDir}{plotname}.png")

plt.show()
